# 🎯 AI-Powered Career Recommendation System
## Using O*NET Database | End-to-End Machine Learning Pipeline

**Pipeline:**
1. Data Loading & Merging
2. Exploratory Data Analysis (EDA)
3. Feature Engineering (3-group vectors)
4. Preprocessing
5. Multi-Label RIASEC Classification (11 models)
6. Model Comparison & Best Model Selection
7. K-Means Career Clustering (Elbow Method)
8. Hybrid Recommendation Engine
9. Skill Gap Analysis
10. SHAP Explainability
11. Career Transition Path


## 0. Setup & Imports

In [30]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.append("..")   # so we can import from src/ and config.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Project modules
from config import *
from src.utils.data_loader import load_all_data, build_occupation_meta, build_riasec_labels
from src.utils.feature_engineering import (
    build_numeric_features, build_tfidf_features,
    build_work_activity_features, build_full_feature_matrix
)
from src.models.classifiers import run_full_model_comparison
from src.models.clustering import find_optimal_k, fit_kmeans, label_clusters, plot_clusters_2d
from src.engine.recommender import HybridCareerRecommender
from src.engine.explainability import (
    compute_shap_values, plot_shap_summary,
    get_top_features_for_user, generate_text_explanation
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
print("✅ All imports successful!")
print(f"📁 Data directory: {DATA_RAW}")


✅ All imports successful!
📁 Data directory: D:\adity\career_recommendation_system\data\raw


---
## 1. Data Loading & Merging

In [31]:
# ── Load all 9 O*NET files ──────────────────────────────────────────────────
data = load_all_data()

# ── Build occupation metadata ───────────────────────────────────────────────
occupation_meta = build_occupation_meta(data)
print(f"\n📋 Occupation Meta:\n{occupation_meta.head(3)}")
print(f"\nShape: {occupation_meta.shape}")



📂 Loading O*NET Database Files...
  ✅ Loading Occupation Data.xlsx...
  ✅ Loading Skills.xlsx...
  ✅ Loading Abilities.xlsx...
  ✅ Loading Knowledge.xlsx...
  ✅ Loading Interests.xlsx...
  ✅ Loading Work Activities.xlsx...
  ✅ Loading Work Styles.xlsx...
  ✅ Loading Job Zones.xlsx...
  ✅ Loading Task Statements.xlsx...

✅ All files loaded successfully!

  📋 Occupation meta: 1016 occupations

📋 Occupation Meta:
                                      title  \
soc_code                                      
11-1011.00                 Chief Executives   
11-1011.03    Chief Sustainability Officers   
11-1021.00  General and Operations Managers   

                                                  description  job_zone  
soc_code                                                                 
11-1011.00  Determine and formulate policies and provide o...         5  
11-1011.03  Communicate and coordinate with management, sh...         5  
11-1021.00  Plan, direct, or coordinate the operation

In [32]:
# ── Build RIASEC multi-label targets ────────────────────────────────────────
riasec_labels = build_riasec_labels(data)
print(f"\n🎯 RIASEC Labels (first 5 rows):")
print(riasec_labels.head())
print(f"\nShape: {riasec_labels.shape}")
print(f"\n📊 RIASEC Score Statistics:")
print(riasec_labels.describe().round(4))


  🎯 RIASEC labels: 923 occupations × 6 RIASEC types

🎯 RIASEC Labels (first 5 rows):
Element Name    Realistic  Investigative  Artistic  Social  Enterprising  \
O*NET-SOC Code                                                             
11-1011.00         0.0574         0.1390    0.0985  0.1613        0.3172   
11-1011.03         0.0849         0.1990    0.1032  0.1478        0.2781   
11-1021.00         0.1019         0.1098    0.0598  0.1566        0.3244   
11-1031.00         0.0754         0.1641    0.1322  0.1807        0.2703   
11-2011.00         0.0473         0.0800    0.1832  0.1486        0.3313   

Element Name    Conventional  
O*NET-SOC Code                
11-1011.00            0.2265  
11-1011.03            0.1869  
11-1021.00            0.2475  
11-1031.00            0.1773  
11-2011.00            0.2097  

Shape: (923, 6)

📊 RIASEC Score Statistics:
Element Name  Realistic  Investigative  Artistic   Social  Enterprising  \
count          923.0000       923.0000  923.0

---
## 2. Exploratory Data Analysis (EDA)

In [33]:
# ── 2.1 RIASEC Distribution Across All Occupations ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c"]

for i, (riasec, color) in enumerate(zip(RIASEC_CODES, colors)):
    axes[i].hist(riasec_labels[riasec], bins=40, color=color, alpha=0.8, edgecolor="white")
    axes[i].set_title(f"{riasec} Distribution", fontweight="bold")
    axes[i].set_xlabel("Score (normalized)")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(riasec_labels[riasec].mean(), color="black", linestyle="--",
                    label=f"mean={riasec_labels[riasec].mean():.3f}")
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle("RIASEC Score Distributions Across All Occupations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_riasec_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ RIASEC distributions plotted")


✅ RIASEC distributions plotted


In [34]:
# ── 2.2 RIASEC Correlation Heatmap ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
corr = riasec_labels.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0,
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={"size": 11, "weight": "bold"})
ax.set_title("RIASEC Inter-Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_riasec_correlation.png", dpi=150, bbox_inches="tight")
plt.show()


In [35]:
# ── 2.3 Dominant RIASEC per Occupation (Bar Chart) ─────────────────────────
dominant_type = riasec_labels.idxmax(axis=1)
counts = dominant_type.value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values,
              color=["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c"])
ax.set_title("Dominant RIASEC Type per Occupation", fontsize=13, fontweight="bold")
ax.set_xlabel("RIASEC Type")
ax.set_ylabel("Number of Occupations")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha="center", va="bottom", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_dominant_riasec.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📊 Dominant RIASEC counts:\n{counts}")



📊 Dominant RIASEC counts:
Realistic        388
Conventional     166
Social           129
Investigative    112
Enterprising     100
Artistic          28
Name: count, dtype: int64


In [36]:
# ── 2.4 Job Zone Distribution ───────────────────────────────────────────────
jz_counts = occupation_meta["job_zone"].value_counts().sort_index()
jz_labels_map = {
    1: "Zone 1\n(Little prep)",
    2: "Zone 2\n(Some prep)",
    3: "Zone 3\n(Medium prep)",
    4: "Zone 4\n(Considerable prep)",
    5: "Zone 5\n(Extensive prep)"
}

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([jz_labels_map.get(k, str(k)) for k in jz_counts.index],
       jz_counts.values, color="#3498db", alpha=0.85, edgecolor="white")
ax.set_title("Occupation Distribution by Job Zone (Experience Level)", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Occupations")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_job_zones.png", dpi=150, bbox_inches="tight")
plt.show()


In [37]:
# ── 2.5 Missing Value Audit & Strategy ─────────────────────────────────────
from src.utils.feature_engineering import audit_missing_values
missing_audit = audit_missing_values(data)

print("""
╔══════════════════════════════════════════════════════════════╗
║  IMPUTATION DECISION: Fill with 0 (not median)              ║
║                                                              ║
║  Reason: O*NET scores range 0-7. A missing score means the  ║
║  skill/ability/activity is absent or not applicable for      ║
║  that occupation — NOT that the value is unknown.            ║
║                                                              ║
║  Filling with MEDIAN would falsely give every occupation a   ║
║  'moderate' score in skills they never use (e.g. a fisherman ║
║  getting a median score in 'Computer Programming').          ║
║                                                              ║
║  Filling with 0 correctly says: 'this occupation does not    ║
║  require this skill' — preserving the semantic meaning.      ║
╚══════════════════════════════════════════════════════════════╝
""")



📊 MISSING VALUE AUDIT (pre-imputation)
─────────────────────────────────────────────────────────────────
  File                    Total Cells    Missing        %  Strategy
─────────────────────────────────────────────────────────────────
  occupation                    3,048          0    0.00%  No action (0% missing)
  skills                      938,700     31,290    3.33%  Fill 0 — skill absent/not required
  abilities                 1,394,640     46,488    3.33%  Fill 0 — ability absent/not required
  knowledge                   885,060     90,608   10.24%  Fill 0 — domain not applicable
  interests                    74,763          0    0.00%  No action (0% missing)
  work_activities           1,099,620    110,900   10.09%  Fill 0 — activity not performed
  work_styles                 336,798          0    0.00%  No action (0% missing)
  job_zones                     4,615          0    0.00%  No action (0% missing)
  tasks                       150,368      2,035    1.35%  Dr

In [38]:
# ── 2.6 Top 20 Highest Paying Occupations by Job Zone ──────────────────────
top_jz5 = occupation_meta[occupation_meta["job_zone"] == 5]["title"].head(20)
print("🏆 Top 20 Zone-5 (Highest Expertise) Occupations:")
for i, title in enumerate(top_jz5, 1):
    print(f"  {i:2d}. {title}")


🏆 Top 20 Zone-5 (Highest Expertise) Occupations:
   1. Chief Executives
   2. Chief Sustainability Officers
   3. Investment Fund Managers
   4. Education Administrators, Kindergarten through Secondary
   5. Education Administrators, Postsecondary
   6. Architectural and Engineering Managers
   7. Natural Sciences Managers
   8. Financial Quantitative Analysts
   9. Health Informatics Specialists
  10. Computer and Information Research Scientists
  11. Mathematicians
  12. Operations Research Analysts
  13. Statisticians
  14. Biostatisticians
  15. Architects, Except Landscape and Naval
  16. Human Factors Engineers and Ergonomists
  17. Microsystems Engineers
  18. Nanosystems Engineers
  19. Animal Scientists
  20. Soil and Plant Scientists


---
## 3. Feature Engineering (3-Group Vectors)

In [39]:
# ── Build full feature matrix ───────────────────────────────────────────────
X, scaler, tfidf_vectorizer = build_full_feature_matrix(data, fit_scaler=True)

# Align RIASEC labels with feature matrix
common_socs = X.index.intersection(riasec_labels.index)
X = X.loc[common_socs]
y = riasec_labels.loc[common_socs]

print(f"\n✅ Final Dataset:")
print(f"   X (features): {X.shape}")
print(f"   y (RIASEC):   {y.shape}")
print(f"   Common SOCs:  {len(common_socs)}")

# Save for use by engine
joblib.dump(X, MASTER_FEATURES_FILE)
joblib.dump(y, MASTER_LABELS_FILE)
joblib.dump(occupation_meta, OCCUPATION_META_FILE)
print("\n💾 Feature matrix saved.")



🔧 Building Full Feature Matrix (3 Groups)...

  [Group 1] Building numeric features (Skills + Abilities + Knowledge + Styles)...
    ✅ Imputed with 0 — skills:0 | abilities:0 | knowledge:0 | work_styles:0 missing values fixed
    ✅ Post-join: 4152 NaN from outer join → filled with 0
    → Shape after imputation: (923, 141)  |  Missing remaining: 0

  [Group 2] Building TF-IDF features from Task Statements...
    → TF-IDF shape: (923, 100)

  [Group 3] Building Work Activity features...
    ✅ Imputed 0 missing values with 0 (activity not performed)
    → Work Activities shape: (894, 41)  |  Missing remaining: 0

  📊 Combined feature matrix: (894, 282)
     → Group 1 (Skills+Abil+Know+Styles): 141 features
     → Group 2 (TF-IDF):                  100 features
     → Group 3 (Work Activities):          41 features

✅ Feature matrix ready: (894, 282)

✅ Final Dataset:
   X (features): (894, 282)
   y (RIASEC):   (894, 6)
   Common SOCs:  894

💾 Feature matrix saved.


In [40]:
# ── Feature Group Summary ───────────────────────────────────────────────────
groups = {
    "Skills + Abilities + Knowledge + Styles":
        [c for c in X.columns if any(c.startswith(p) for p in ["skill__","abil__","know__","style__"])],
    "TF-IDF (Task Statements)":
        [c for c in X.columns if c.startswith("tfidf__")],
    "Work Activities":
        [c for c in X.columns if c.startswith("wa__")],
}
print("📊 Feature Group Breakdown:")
for name, cols in groups.items():
    print(f"  {name}: {len(cols)} features")
print(f"  TOTAL: {X.shape[1]} features")


📊 Feature Group Breakdown:
  Skills + Abilities + Knowledge + Styles: 141 features
  TF-IDF (Task Statements): 100 features
  Work Activities: 41 features
  TOTAL: 282 features


In [41]:
# ── Top Skill Features by Variance ─────────────────────────────────────────
skill_cols = [c for c in X.columns if c.startswith("skill__")]
variances  = X[skill_cols].var().nlargest(15)
clean_names = [c.replace("skill__","") for c in variances.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(clean_names[::-1], variances.values[::-1], color="#3498db", alpha=0.85)
ax.set_title("Top 15 Skills by Variance (Most Discriminating)", fontsize=13, fontweight="bold")
ax.set_xlabel("Variance")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_top_skill_variance.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 4. Preprocessing & Train-Test Split

In [42]:
# ── Train / Test Split ──────────────────────────────────────────────────────
# IMPORTANT: Convert to numpy BEFORE splitting so models train without
# feature names. At inference time we also convert to numpy → no warnings.
feature_cols = list(X.columns)
joblib.dump(feature_cols, DATA_PROCESSED / "feature_cols.pkl")  # save for app

X_np = X.values   # numpy array — no feature names
y_np = y.values

X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=RANDOM_STATE
)

# Keep DataFrame versions only for EDA / SHAP (not for model training)
X_train_df = pd.DataFrame(X_train, columns=feature_cols)
X_test_df  = pd.DataFrame(X_test,  columns=feature_cols)

print(f"✅ Train set: {X_train.shape}  (numpy array — no feature name warnings)")
print(f"✅ Test  set: {X_test.shape}")
print(f"\n📊 RIASEC label statistics (train):")
print(pd.DataFrame(y_train, columns=RIASEC_CODES).describe().round(4))


✅ Train set: (715, 282)  (numpy array — no feature name warnings)
✅ Test  set: (179, 282)

📊 RIASEC label statistics (train):
       Realistic  Investigative  Artistic   Social  Enterprising  Conventional
count   715.0000       715.0000  715.0000 715.0000      715.0000      715.0000
mean      0.2398         0.1618    0.0941   0.1392        0.1377        0.2276
std       0.1184         0.0696    0.0532   0.0728        0.0756        0.0565
min       0.0427         0.0479    0.0429   0.0480        0.0462        0.0763
25%       0.1394         0.1075    0.0577   0.0777        0.0782        0.1902
50%       0.2303         0.1506    0.0761   0.1184        0.1098        0.2271
75%       0.3520         0.2160    0.1145   0.1861        0.1818        0.2617
max       0.4642         0.3316    0.3393   0.3368        0.3458        0.4340


---
## 5. Multi-Output Regression — All Models

> **Why Regression, not Classification?**
> Our RIASEC target is a *continuous probability distribution* per occupation (e.g. Investigative=0.42, Conventional=0.28...). Classifiers like Logistic Regression expect *discrete class labels* and will throw `Unknown label type: continuous`. The correct approach is **Multi-Output Regression** — predict all 6 RIASEC scores simultaneously as real numbers. We evaluate with R², RMSE, and Cosine Similarity between predicted and actual distributions.

In [43]:
# ── Run Full Model Comparison ───────────────────────────────────────────────
leaderboard, trained_models, all_results, best_name, best_model = run_full_model_comparison(
    X_train, y_train, X_test, y_test
)

print("\n🏆 FINAL LEADERBOARD:")
print(leaderboard.to_string())



🏆 FULL MODEL COMPARISON — Simple → Complex

  [1/13] Training: Ridge Regression...
    R²=0.8486  RMSE=0.0270  CosSim=0.9901

  [2/13] Training: Linear Regression...
    R²=0.8126  RMSE=0.0298  CosSim=0.9878

  [3/13] Training: Lasso Regression...
    R²=0.7797  RMSE=0.0324  CosSim=0.9860

  [4/13] Training: KNN Regressor...
    R²=0.8033  RMSE=0.0310  CosSim=0.9870

  [5/13] Training: Decision Tree...
    R²=0.6353  RMSE=0.0430  CosSim=0.9774

  [6/13] Training: Extra Trees...
    R²=0.8345  RMSE=0.0286  CosSim=0.9894

  [7/13] Training: Random Forest...
    R²=0.8133  RMSE=0.0303  CosSim=0.9882

  [8/13] Training: Neural Network (MLP)...
    R²=0.7443  RMSE=0.0355  CosSim=0.9861

  [9/13] Training: Bagging (DT base)...
    R²=0.8079  RMSE=0.0307  CosSim=0.9879

  [10/13] Training: AdaBoost...
    R²=0.7682  RMSE=0.0337  CosSim=0.9855

  [11/13] Training: Gradient Boosting...
    R²=0.8277  RMSE=0.0291  CosSim=0.9892

  [12/13] Training: XGBoost...
    R²=0.8448  RMSE=0.0278  CosSim=

---
## 6. Model Comparison & Visualization

In [44]:
# ── 6.1 R² Score Comparison Bar Chart ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Model Comparison: Multi-Label RIASEC Prediction", fontsize=14, fontweight="bold")

colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(leaderboard)))

# R² Score
axes[0].barh(leaderboard["Model"][::-1], leaderboard["R² Score"][::-1], color=colors_bar)
axes[0].set_title("R² Score (higher = better)", fontweight="bold")
axes[0].set_xlabel("R² Score")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].grid(True, alpha=0.3, axis="x")

# RMSE
colors_rmse = plt.cm.RdYlGn(np.linspace(0.9, 0.2, len(leaderboard)))
axes[1].barh(leaderboard["Model"][::-1], leaderboard["RMSE"][::-1], color=colors_rmse)
axes[1].set_title("RMSE (lower = better)", fontweight="bold")
axes[1].set_xlabel("RMSE")
axes[1].grid(True, alpha=0.3, axis="x")

# Cosine Similarity
axes[2].barh(leaderboard["Model"][::-1], leaderboard["Cosine Sim"][::-1], color=colors_bar)
axes[2].set_title("Cosine Similarity (higher = better)", fontweight="bold")
axes[2].set_xlabel("Cosine Similarity")
axes[2].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [45]:
# ── 6.2 Per-RIASEC R² Heatmap for Best Model ────────────────────────────────
best_result = next(r for r in all_results if r["Model"] == best_name)
per_riasec  = pd.DataFrame([best_result["Per-RIASEC R²"]], index=[best_name])

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(per_riasec, annot=True, fmt=".4f", cmap="YlGn",
            ax=ax, linewidths=0.5, cbar_kws={"label": "R² Score"})
ax.set_title(f"Per-RIASEC R² Score — Best Model: {best_name}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "best_model_per_riasec_r2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n🥇 Best Model: {best_name}")
print(f"   Overall R²:   {best_result['R² Score']}")
print(f"   RMSE:         {best_result['RMSE']}")
print(f"   Cosine Sim:   {best_result['Cosine Sim']}")



🥇 Best Model: Ridge Regression
   Overall R²:   0.8486
   RMSE:         0.027
   Cosine Sim:   0.9901


In [46]:
# ── 6.3 Prediction vs Actual scatter for best model ─────────────────────────
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (riasec, ax) in enumerate(zip(RIASEC_CODES, axes)):
    ax.scatter(y_test[:, i], y_pred_best[:, i], alpha=0.3, s=10, c="#3498db")
    max_val = max(y_test[:, i].max(), y_pred_best[:, i].max())
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Perfect")
    ax.set_title(f"{riasec}", fontweight="bold")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Predicted vs Actual RIASEC Scores — {best_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "pred_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 7. K-Means Career Clustering

In [47]:
# ── 7.1 Elbow Method ────────────────────────────────────────────────────────
# Use work activities + skills for clustering
cluster_cols = [c for c in X.columns if c.startswith("wa__") or c.startswith("skill__")]
X_cluster    = X[cluster_cols].values

optimal_k, inertias, silhouettes, elbow_fig = find_optimal_k(X_cluster, max_k=KMEANS_MAX_K)
elbow_fig



🔍 Finding Optimal K via Elbow Method + Silhouette Score...
  K= 2  Inertia=1714.0  Silhouette=0.2697
  K= 3  Inertia=1511.9  Silhouette=0.2273
  K= 4  Inertia=1369.2  Silhouette=0.1621
  K= 5  Inertia=1279.5  Silhouette=0.1555
  K= 6  Inertia=1201.4  Silhouette=0.1557
  K= 7  Inertia=1143.9  Silhouette=0.1337
  K= 8  Inertia=1100.8  Silhouette=0.1318
  K= 9  Inertia=1072.5  Silhouette=0.1218
  K=10  Inertia=1040.0  Silhouette=0.1256
  K=11  Inertia=1021.8  Silhouette=0.1112
  K=12  Inertia=999.2  Silhouette=0.1180
  K=13  Inertia=979.5  Silhouette=0.1143
  K=14  Inertia=963.1  Silhouette=0.1196
  K=15  Inertia=945.0  Silhouette=0.1073

  ✅ Elbow suggests K=4, Silhouette suggests K=2
  🏆 Selected K=2 (best silhouette)


<Figure size 1400x500 with 2 Axes>

In [48]:
# ── 7.2 Fit Final K-Means ────────────────────────────────────────────────────
km_model, cluster_series = fit_kmeans(X_cluster, k=optimal_k, soc_index=X.index)

# Label clusters semantically using RIASEC profiles
cluster_labels_dict, cluster_summaries = label_clusters(
    km_model, X_cluster, X.index, riasec_labels, occupation_meta
)
print(f"\n✅ Cluster labels: {cluster_labels_dict}")



🔧 Fitting K-Means with K=2...
  Cluster  0: Finance & Administration        (dominant=Conventional, n=507, sample=['Chief Executives', 'Chief Sustainability Officers'])
  Cluster  1: Engineering & Trades            (dominant=Realistic, n=387, sample=['Farm Labor Contractors', 'Computer Network Support Specialists'])

✅ Cluster labels: {0: 'Finance & Administration', 1: 'Engineering & Trades'}


In [49]:
# ── 7.3 2D PCA Visualization ─────────────────────────────────────────────────
plot_clusters_2d(X_cluster, km_model.labels_, cluster_labels_dict,
                  title=f"Career Clusters (K={optimal_k}) — PCA 2D")
plt.figure(figsize=(1,1))
img = plt.imread(REPORTS_DIR / "cluster_visualization.png")
plt.figure(figsize=(12,8))
plt.imshow(img); plt.axis("off")
plt.show()


  📊 Cluster plot saved.


In [50]:
# ── 7.4 Cluster Size Distribution ────────────────────────────────────────────
cluster_counts = cluster_series.value_counts().sort_index()
cluster_names  = [cluster_labels_dict.get(i, f"Cluster {i}") for i in cluster_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(cluster_names, cluster_counts.values,
              color=plt.cm.Set3(np.linspace(0, 1, len(cluster_counts))))
ax.set_title(f"Career Cluster Sizes (K={optimal_k})", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Occupations")
plt.xticks(rotation=30, ha="right")
for bar, val in zip(bars, cluster_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha="center", fontweight="bold", fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "cluster_sizes.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 8. SHAP Explainability

In [51]:
# ── Compute SHAP on test sample ──────────────────────────────────────────────
# Use 100 samples for speed — increase if needed
X_shap_sample = X_test_df.iloc[:100]

try:
    shap_explainer, shap_values = compute_shap_values(
        best_model, X_train_df, X_shap_sample, max_samples=100
    )
    # Plot for Investigative (index 1) — most common in data science careers
    plot_shap_summary(shap_values, X_shap_sample, riasec_idx=1, top_n=15)
    plot_shap_summary(shap_values, X_shap_sample, riasec_idx=0, top_n=15)
    print("\n✅ SHAP plots saved to reports/")
    SHAP_AVAILABLE = True
except Exception as e:
    print(f"⚠️  SHAP computation skipped: {e}")
    print("    (SHAP works best with tree-based models. Proceeding without it.)")
    SHAP_AVAILABLE = False
    shap_explainer, shap_values = None, None



🔍 Computing SHAP values...
  ✅ Used LinearExplainer

  📊 SHAP summary for: Investigative
  💾 Saved: shap_investigative.png

  📊 SHAP summary for: Realistic
  💾 Saved: shap_realistic.png

✅ SHAP plots saved to reports/


---
## 9. Hybrid Recommendation Engine — Live Demo

In [52]:
# ── Build the recommender ────────────────────────────────────────────────────
recommender = HybridCareerRecommender(
    best_model        = best_model,
    job_vectors       = X,
    riasec_labels     = riasec_labels,
    occupation_meta   = occupation_meta,
    cluster_labels    = cluster_labels_dict,
    cluster_assignments = cluster_series,
    feature_columns   = list(X.columns),
    scaler            = scaler,
)
print("✅ Recommender built and ready!")


✅ Recommender built and ready!


In [53]:
# ── Demo: Define a User Profile ──────────────────────────────────────────────
# Simulating a user who is good at data/analytics and interested in research

USER_SKILLS = {
    "Programming":             4,
    "Mathematics":             5,
    "Critical Thinking":       5,
    "Data Analysis":           4,
    "Writing":                 3,
    "Communication":           3,
    "Problem Solving":         5,
    "Research":                4,
}

USER_INTERESTS = {
    "Investigative": 5,
    "Conventional":  3,
    "Realistic":     2,
    "Artistic":      1,
    "Social":        2,
    "Enterprising":  3,
}

# ── Vectorize user ─────────────────────────────────────────────────────────
from src.utils.feature_engineering import vectorize_user_profile

user_vector, user_riasec_norm = vectorize_user_profile(
    user_skills       = USER_SKILLS,
    user_interests    = USER_INTERESTS,
    all_feature_columns = list(X.columns),
    scaler            = scaler,
    riasec_codes      = RIASEC_CODES
)
print("✅ User profile vectorized!")
print(f"   User vector shape: {user_vector.shape}")
print(f"   User RIASEC (normalized): {user_riasec_norm}")


✅ User profile vectorized!
   User vector shape: (1, 282)
   User RIASEC (normalized): {'Realistic': 0.125, 'Investigative': 0.3125, 'Artistic': 0.0625, 'Social': 0.125, 'Enterprising': 0.1875, 'Conventional': 0.1875}


In [54]:
# ── Run Recommendation ───────────────────────────────────────────────────────
results = recommender.recommend(
    user_vector       = user_vector,
    user_riasec_input = USER_INTERESTS,
    top_n             = 10,
    current_career_soc= None    # Set a SOC code here for transition path
)

print(f"\n{'='*65}")
print(f"🎯 YOUR RIASEC PROFILE")
print(f"{'='*65}")
for code, score in sorted(results['riasec_profile'].items(),
                            key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 30)
    print(f"  {code:15s} {score*100:5.1f}%  {bar}")

print(f"\n{'='*65}")
print(f"🏆 TOP {len(results['recommendations'])} CAREER RECOMMENDATIONS")
print(f"{'='*65}")
for r in results["recommendations"]:
    print(f"\n  #{r['rank']} {r['title']}")
    print(f"     Match Score:    {r['match_score']}%")
    print(f"     Career Domain:  {r['career_domain']}")
    print(f"     Job Zone:       {r['job_zone']}/5")
    if r['skill_gaps']:
        gaps = ", ".join([g[0] for g in r['skill_gaps'][:3]])
        print(f"     Skill Gaps:     {gaps}")



🚀 Running Hybrid Recommendation Engine...
  📊 RIASEC Profile: Realistic=19.6% | Investigative=31.3% | Artistic=17.7% | Social=11.9% | Enterprising=12.0% | Conventional=7.5%
  🎯 Dominant types: ['Investigative', 'Realistic'] → 785 candidate careers

🎯 YOUR RIASEC PROFILE
  Investigative    31.3%  █████████
  Realistic        19.6%  █████
  Artistic         17.7%  █████
  Enterprising     12.0%  ███
  Social           11.9%  ███
  Conventional      7.5%  ██

🏆 TOP 10 CAREER RECOMMENDATIONS

  #1 Social Science Research Assistants
     Match Score:    -7.9%
     Career Domain:  Finance & Administration
     Job Zone:       4/5
     Skill Gaps:     Oral Comprehension, Information Ordering, Oral Expression

  #2 Mathematicians
     Match Score:    -8.7%
     Career Domain:  Finance & Administration
     Job Zone:       5/5
     Skill Gaps:     Information Ordering, Oral Comprehension, Deductive Reasoning

  #3 Astronomers
     Match Score:    -9.0%
     Career Domain:  Finance & Administra

In [55]:
# ── Visualize Top Recommendations ───────────────────────────────────────────
recs   = results["recommendations"]
titles = [r["title"][:35] + "..." if len(r["title"]) > 35 else r["title"] for r in recs]
scores = [r["match_score"] for r in recs]
colors_rec = plt.cm.RdYlGn(np.linspace(0.4, 0.9, len(recs)))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(titles[::-1], scores[::-1], color=colors_rec)
ax.set_title("Top Career Recommendations — Match Score (%)", fontsize=13, fontweight="bold")
ax.set_xlabel("Match Score (%)")
ax.set_xlim(0, 105)
for bar, score in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{score}%", va="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "top_recommendations.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 10. Skill Gap Analysis — Deep Dive

In [56]:
# ── Skill Gap for Top 3 Careers ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Skill Gap Analysis — Top 3 Recommended Careers", fontsize=13, fontweight="bold")

for idx in range(min(3, len(recs))):
    r    = recs[idx]
    gaps = r["skill_gaps"][:8]
    if not gaps:
        axes[idx].text(0.5, 0.5, "No significant gaps!", ha="center", va="center")
        axes[idx].set_title(r["title"][:30])
        continue
    skills_g, scores_g = zip(*gaps)
    axes[idx].barh(list(skills_g)[::-1], list(scores_g)[::-1], color="#e74c3c", alpha=0.85)
    axes[idx].set_title(r["title"][:30] + f"\n({r['match_score']}% match)", fontweight="bold")
    axes[idx].set_xlabel("Gap Score")
    axes[idx].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "skill_gap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 11. Career Transition Path — Bonus Feature

In [57]:
# ── Career Transition Demo ───────────────────────────────────────────────────
# Pick top recommended career as target
top_career_soc = recs[0]["soc_code"]

# Find a "current" career — e.g., pick a Zone-2 occupation as starting point
zone2_socs = occupation_meta[occupation_meta["job_zone"] == 2].index
zone2_socs = zone2_socs.intersection(X.index)
current_soc = zone2_socs[0] if len(zone2_socs) > 0 else None

if current_soc:
    transition = recommender.career_transition_path(current_soc, top_career_soc, user_vector)
    print(f"\n{'='*65}")
    print(f"🛤️  CAREER TRANSITION PATH")
    print(f"{'='*65}")
    print(f"  FROM:  {transition['current']}")
    print(f"  TO:    {transition['target']}")
    print(f"  Feasibility: {transition['feasibility']}")
    print(f"\n  📚 Skills to Learn:")
    for skill in transition['skills_to_learn']:
        print(f"     • {skill}")
    if transition["intermediate_roles"]:
        print(f"\n  🪜 Intermediate Stepping-Stone Roles:")
        for role in transition["intermediate_roles"]:
            print(f"     → {role}")
    print(f"\n  📈 Job Zone Change: {transition['job_zone_change']:+d} levels")
else:
    print("⚠️  No Zone-2 occupations found in dataset.")



🛤️  CAREER TRANSITION PATH
  FROM:  Food Service Managers
  TO:    Social Science Research Assistants
  Feasibility: 🔴 Challenging Transition (requires significant upskilling)

  📚 Skills to Learn:
     • Science
     • Programming
     • English Language
     • Technology Design
     • Information Ordering
     • Writing
     • Reading Comprehension

  🪜 Intermediate Stepping-Stone Roles:
     → First-Line Supervisors of Food Preparation and Serving Workers
     → Chefs and Head Cooks
     → First-Line Supervisors of Retail Sales Workers

  📈 Job Zone Change: +2 levels


---
## 12. Final Conclusion & Model Summary

In [58]:
print(f"\n{'='*65}")
print(f"🏆 FINAL CONCLUSION")
print(f"{'='*65}")
print(f"\n  Best Model:     {best_name}")
best_r = next(r for r in all_results if r["Model"] == best_name)
print(f"  R² Score:       {best_r['R² Score']:.4f}")
print(f"  RMSE:           {best_r['RMSE']:.4f}")
print(f"  Cosine Sim:     {best_r['Cosine Sim']:.4f}")

print(f"\n  📊 Model Rankings:")
print(leaderboard.to_string())

print(f"\n  📁 Deliverables:")
print(f"     ✅ Jupyter Notebook   — Full EDA + ML Pipeline")
print(f"     ✅ Modular Backend    — src/ modules")
print(f"     ✅ Streamlit UI       — streamlit_app/")
print(f"     ✅ Saved Models       — outputs/models/")
print(f"     ✅ Reports & Plots    — outputs/reports/")

print(f"\n  🎯 System Capabilities:")
print(f"     ✅ Multi-Label RIASEC Prediction")
print(f"     ✅ Hybrid Recommendation (Classifier + Cosine Similarity)")
print(f"     ✅ Skill Gap Analysis")
print(f"     ✅ SHAP Explainability")
print(f"     ✅ Career Transition Path")
print(f"     ✅ K-Means Career Domain Clustering (K={optimal_k})")



🏆 FINAL CONCLUSION

  Best Model:     Ridge Regression
  R² Score:       0.8486
  RMSE:           0.0270
  Cosine Sim:     0.9901

  📊 Model Rankings:
                   Model  R² Score   RMSE  Cosine Sim
1       Ridge Regression    0.8486 0.0270      0.9901
2                XGBoost    0.8448 0.0278      0.9900
3               LightGBM    0.8421 0.0280      0.9899
4            Extra Trees    0.8345 0.0286      0.9894
5      Gradient Boosting    0.8277 0.0291      0.9892
6          Random Forest    0.8133 0.0303      0.9882
7      Linear Regression    0.8126 0.0298      0.9878
8      Bagging (DT base)    0.8079 0.0307      0.9879
9          KNN Regressor    0.8033 0.0310      0.9870
10      Lasso Regression    0.7797 0.0324      0.9860
11              AdaBoost    0.7682 0.0337      0.9855
12  Neural Network (MLP)    0.7443 0.0355      0.9861
13         Decision Tree    0.6353 0.0430      0.9774

  📁 Deliverables:
     ✅ Jupyter Notebook   — Full EDA + ML Pipeline
     ✅ Modular Backend